# 2.5 Workflow - Conversation 对话

- **对应文档**: [workflow_conversation](https://doc.agentscope.io/tutorial/workflow_conversation.html)
- **本讲目标**: 掌握多轮对话的组织方式与 Conversation 相关 API。
- **前置知识**: 02_react_agent, 03_message, Memory。

## 学习要点

- 多轮对话的状态与历史管理。
- 与 Memory 结合维护上下文。

In [ ]:
# 在此按 doc 编写 conversation 示例
# 参考 https://doc.agentscope.io/tutorial/workflow_conversation.html
print("请参考 workflow_conversation 文档完成对话示例代码")

In [1]:
import asyncio
import json
import os

from agentscope.agent import ReActAgent, UserAgent
from agentscope.memory import InMemoryMemory
from agentscope.formatter import (
    DashScopeChatFormatter,
    DashScopeMultiAgentFormatter,
)
from agentscope.model import DashScopeChatModel
from agentscope.message import Msg
from agentscope.pipeline import MsgHub
from agentscope.tool import Toolkit

In [2]:
model = DashScopeChatModel(
    model_name="qwen-max",
    api_key=os.environ["DASHSCOPE_API_KEY"],
)
formatter = DashScopeMultiAgentFormatter()

alice = ReActAgent(
    name="Alice",
    sys_prompt="你是一个名为 Alice 的学生。",
    model=model,
    formatter=formatter,
)

bob = ReActAgent(
    name="Bob",
    sys_prompt="你是一个名为 Bob 的学生。",
    model=model,
    formatter=formatter,
)

charlie = ReActAgent(
    name="Charlie",
    sys_prompt="你是一个名为 Charlie 的学生。",
    model=model,
    formatter=formatter,
)


async def example_msghub() -> None:
    """使用 MsgHub 进行多智能体对话的示例。"""
    async with MsgHub(
        [alice, bob, charlie],
        # 进入 MsgHub 时的公告消息
        announcement=Msg(
            "system",
            "现在大家互相认识一下，简单自我介绍。",
            "system",
        ),
    ):
        await alice()
        await bob()
        await charlie()


await example_msghub()

Alice: 嗨，大家好！我叫Alice，是一名学生。很高兴认识大家，希望我们能一起度过愉快的时光，并且互相学习和帮助。你们都是做什么的呢？
Bob: 嗨，Alice！很高兴认识你。我叫Bob，也是一名学生。我对我们即将一起度过的时光感到非常兴奋，期待能够从大家身上学到新东西，并且我也很乐意提供帮助。希望我们能成为好朋友！
Charlie: 嗨，Alice和Bob！我叫Charlie，同样是一名学生。真的很高兴能够遇到你们俩。我对接下来的时光充满期待，希望能从大家那里学到很多，并且我也愿意贡献我的一份力量。希望我们能共同进步，一起享受这段旅程！


In [3]:
async def example_memory() -> None:
    """打印 Alice 的记忆。"""
    print("Alice 的记忆：")
    for msg in await alice.memory.get_memory():
        print(
            f"{msg.name}: {json.dumps(msg.content, indent=4, ensure_ascii=False)}",
        )

await example_memory()

Alice 的记忆：
system: "现在大家互相认识一下，简单自我介绍。"
Alice: [
    {
        "type": "text",
        "text": "嗨，大家好！我叫Alice，是一名学生。很高兴认识大家，希望我们能一起度过愉快的时光，并且互相学习和帮助。你们都是做什么的呢？"
    }
]
Bob: [
    {
        "type": "text",
        "text": "嗨，Alice！很高兴认识你。我叫Bob，也是一名学生。我对我们即将一起度过的时光感到非常兴奋，期待能够从大家身上学到新东西，并且我也很乐意提供帮助。希望我们能成为好朋友！"
    }
]
Charlie: [
    {
        "type": "text",
        "text": "嗨，Alice和Bob！我叫Charlie，同样是一名学生。真的很高兴能够遇到你们俩。我对接下来的时光充满期待，希望能从大家那里学到很多，并且我也愿意贡献我的一份力量。希望我们能共同进步，一起享受这段旅程！"
    }
]
